In [1]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
import glob
import os
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import gradio as gr


/Users/prasanthravula/LLM_ENGINEERING_PROJECTS/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/prasanthravula/LLM_ENGINEERING_PROJECTS/.venv/lib/python3.14/site-packages/multiprocess/connection.py:335: SyntaxWarning: 'return' in a 'finally' block
  return f
/Users/prasanthravula/LLM_ENGINEERING_PROJECTS/.venv/lib/python3.14/site-packages/multiprocess/connection.py:337: SyntaxWarning: 'return' in a 'finally' block
  return self._get_more_data(ov, maxsize)


In [2]:
db_name = "vector_db"
model_name = "kimi-k2.5:cloud" #gemma:latest

In [3]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
retriever = vectorstore.as_retriever() #search_kwargs={"k": 3}
llm = ChatOllama(temperature=0, model=model_name)   

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='963c3bf4-ea74-42fc-9c5a-b08e10eb335e', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [6]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

answer_question("Who is Averi Lancaster?", [])

'I believe you\'re referring to **Avery Lancaster** (spelled with an "e"), who is the Co-Founder and Chief Executive Officer (CEO) of Insurellm.\n\nHere are the key details about Avery:\n\n**Current Role:**\n- **Co-Founder & CEO** at Insurellm (2015–Present)\n- Based in San Francisco, California\n- Current salary: $225,000\n\n**Background:**\nAvery co-founded Insurellm in 2015 and has been instrumental in growing the company into a leading Insurance Tech provider. She is known for her innovative leadership strategies and expertise in risk management, which have helped catapult the company into the mainstream insurance market.\n\n**Prior Experience:**\nBefore launching Insurellm, Avery served as a Senior Product Manager at Innovate Insurance Solutions (2013–2015), where she developed groundbreaking insurance products specifically aimed at the tech sector.\n\nIs there anything specific about Avery\'s role at Insurellm or her background that you\'d like to know more about?'

In [7]:

gr.ChatInterface(answer_question).launch()

/Users/prasanthravula/LLM_ENGINEERING_PROJECTS/.venv/lib/python3.14/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
